# ThessLink RL — Training Notebook

Εκπαίδευση και αξιολόγηση των αλγορίθμων **Q-Learning**, **DQN**, **PPO** σε grids 8×8, 32×32, 64×64.

**Action space:** `MultiDiscrete([5, 5])` — movement only, no explicit target selection. The reward function guides agents toward the cost-optimal POI.

| Grid | Q-Learning | DQN | PPO |
|------|-----------|-----|-----|
| 8×8  | `nav_qtable_8.pkl` | `nav_dqn_8.zip` | `nav_ppo_8.zip` |
| 32×32 | `nav_qtable_32.pkl` | `nav_dqn_32.zip` | `nav_ppo_32.zip` |
| 64×64 | `nav_qtable_64.pkl` | `nav_dqn_64.zip` | `nav_ppo_64.zip` |

In [ ]:
import sys
from pathlib import Path

# Make sure the project root and lb-foraging are on the path
ROOT = Path(".").resolve()
LBF = ROOT / "lb-foraging"
for p in [str(ROOT), str(LBF)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", ROOT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from navigation_train import (
    train_ppo, train_dqn, train_qlearning,
    _eval_navigation, _discretize_nav, _NAV_ACTIONS,
    MODEL_DIR, get_history_path,
)

print("Imports OK")

---
## Βοηθητικές συναρτήσεις

In [ ]:
import pickle

GRID_SIZES = [8]
ALGOS = ["ppo", "dqn", "qlearning"]


def get_plot_path(algo: str, grid_size: int | str):
    """Path to training_plot_*.png in the model folder."""
    return MODEL_DIR / algo / f"training_plot_{algo}_{str(grid_size)}.png"


def model_exists(algo: str, grid_size: int) -> bool:
    """Check if a trained model file exists."""
    tag = str(grid_size)
    if algo == "ppo":
        return (MODEL_DIR / "ppo" / f"nav_ppo_{tag}.zip").exists()
    elif algo == "dqn":
        return (MODEL_DIR / "dqn" / f"nav_dqn_{tag}.zip").exists()
    else:
        return (MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl").exists()


def history_exists(algo: str, grid_size: int) -> bool:
    """Check if training_history_*.pkl exists (source for plot generation)."""
    return get_history_path(algo, grid_size).exists()


def show_status():
    """Print a table showing which models and plots exist."""
    header = f"{'':12}" + "".join(f"{g:>8}" for g in GRID_SIZES)
    print(header)
    print("-" * (12 + 8 * len(GRID_SIZES)))
    for algo in ALGOS:
        row = f"{algo:<12}"
        for g in GRID_SIZES:
            m = "M" if model_exists(algo, g) else "."
            p = "H" if history_exists(algo, g) else "."
            row += f"  [{m}{p}]  "
        print(row)
    print("\nM=model exists  H=history (.pkl) exists  .=missing")


show_status()

---
## Training

**Focus: 8×8 first** — optimize & validate before scaling to 64×64.

**Centralized controller:** A single model sees the full global state (35 floats: relative vectors to POIs + wall bits + cost components × 3 POIs) and outputs a **joint movement** `MultiDiscrete([5, 5])` (or `Discrete(25)` for DQN/Q-Learning). No explicit target selection — the reward function guides agents toward the cost-optimal POI via progress reward + cost-scaled terminal bonus. Agreement = arrived at cost-optimal POI.

Επίλεξε αλγόριθμο και grid size και τρέξε το αντίστοιχο cell.

### PPO

In [ ]:
# 6 parallel envs (SubprocVecEnv), entropy bonus, 128x128 network
_PPO_STEPS = {8: 2_000_000, 32: 2_000_000, 64: 2_000_000}

GRID = 8   # ← αλλαγή: 8, 32, ή 64
STEPS = _PPO_STEPS[GRID]

train_ppo(
    total_timesteps=STEPS,
    seed=42,
    eval_freq=50_000,
    grid_size=(GRID, GRID),
)
show_status()

### DQN

In [ ]:
# 6 parallel envs, 128x128 network, buffer=200k
_DQN_STEPS = {8: 2_000_000, 32: 2_000_000, 64: 2_000_000}

GRID = 8   # ← αλλαγή: 8, 32, ή 64
STEPS = _DQN_STEPS[GRID]

train_dqn(
    total_timesteps=STEPS,
    seed=42,
    eval_freq=50_000,
    grid_size=(GRID, GRID),
)
show_status()

### Q-Learning

In [ ]:
# Compact state (442,368 states) — parallel workers ~6x faster than single-threaded
EPISODES = 800_000
GRID = 8

train_qlearning(
    total_episodes=EPISODES,
    seed=42,
    eval_freq=20_000,   # 40 checkpoints over 800k episodes
    n_workers=6,        # parallel processes; each chunk split across workers
    grid_size=(GRID, GRID),
)
show_status()

---
## Plots (Cumulative)

### Ένα συγκεκριμένο plot

In [ ]:
ALGO = "ppo"   # ← ppo, dqn, qlearning
GRID = 8       # ← 8, 32, 64

def cumulative_avg(values):
    return np.cumsum(values) / np.arange(1, len(values) + 1)

ALGO_CONFIG = {"ppo": ("PPO", "tab:blue"), "dqn": ("DQN", "tab:green"), "qlearning": ("Q-Learning", "tab:orange")}
path = get_history_path(ALGO, GRID)
if path.exists():
    with open(path, "rb") as f:
        data = pickle.load(f)
    title, color = ALGO_CONFIG[ALGO]
    fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    steps = data["steps"]
    cum_reward = cumulative_avg(data["rewards"])
    cum_agreement = cumulative_avg(data["agreement"])
    axes[0].plot(steps, cum_reward, color=color, linewidth=1.5)
    axes[0].set_ylabel("Cumulative Mean Reward")
    axes[0].set_title(f"Training Progress ({title} {GRID}×{GRID})")
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(steps, cum_agreement, color=color, linewidth=1.5)
    axes[1].set_ylabel("Cumulative Agreement Rate")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xlabel("Training Steps / Episodes")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = get_plot_path(ALGO, GRID)
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"History not found: {path}. Run {ALGO} training first.")

### Σύγκριση PPO, DQN, Q-Learning (combined)

Όλα τα τρία μοντέλα σε ένα figure. Απαιτεί `training_history_*.pkl` από προηγούμενα training runs.

In [ ]:
GRID = 8   # ← 8, 32, 64

algo_configs = [("ppo", "PPO", "tab:blue"), ("dqn", "DQN", "tab:green"), ("qlearning", "Q-Learning", "tab:orange")]
histories = {}
for algo, label, color in algo_configs:
    path = get_history_path(algo, GRID)
    if path.exists():
        with open(path, "rb") as f:
            histories[algo] = {"label": label, "color": color, **pickle.load(f)}
    else:
        print(f"  Skipping {label}: {path} not found")

if histories:
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    for algo, data in histories.items():
        steps = data["steps"]
        cum_reward = cumulative_avg(data["rewards"])
        cum_agreement = cumulative_avg(data["agreement"])
        axes[0].plot(steps, cum_reward, color=data["color"], linewidth=1.5, label=data["label"])
        axes[1].plot(steps, cum_agreement, color=data["color"], linewidth=1.5, label=data["label"])
    axes[0].set_ylabel("Cumulative Mean Reward")
    axes[0].set_title(f"Training Progress ({GRID}×{GRID}) — PPO vs DQN vs Q-Learning")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].set_ylabel("Cumulative Agreement Rate")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xlabel("Training Steps / Episodes")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = MODEL_DIR / f"training_plot_combined_{GRID}.png"
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Run PPO, DQN, Q-Learning first to generate training histories.")

---
## Αξιολόγηση (Evaluation)

### Ένα μοντέλο

In [ ]:
ALGO = "ppo"   # ← ppo, dqn, qlearning
GRID = 8       # ← 8, 32, 64
N_EPISODES = 200

tag = str(GRID)
grid_size = (GRID, GRID)
max_steps = max(300, GRID * GRID // 2)

if not model_exists(ALGO, GRID):
    print(f"Model not found: {ALGO} {GRID}×{GRID}. Train it first.")
else:
    if ALGO == "ppo":
        from stable_baselines3 import PPO
        m = PPO.load(str(MODEL_DIR / "ppo" / f"nav_ppo_{tag}"))
        predict = lambda obs: m.predict(obs, deterministic=True)[0]
    elif ALGO == "dqn":
        from stable_baselines3 import DQN
        m = DQN.load(str(MODEL_DIR / "dqn" / f"nav_dqn_{tag}"))
        predict = lambda obs: int(m.predict(obs, deterministic=True)[0])
    else:
        with open(MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl", "rb") as f:
            q_table = pickle.load(f)
        predict = lambda obs: int(np.argmax(q_table.get(_discretize_nav(obs), np.zeros(_NAV_ACTIONS))))

    stats = _eval_navigation(predict, n_episodes=N_EPISODES, grid_size=grid_size, max_steps=max_steps)
    print(f"\n{'='*40}")
    print(f"  {ALGO.upper()} — {GRID}×{GRID} ({N_EPISODES} episodes)")
    print(f"{'='*40}")
    print(f"  Mean reward  : {stats['mean_reward']:.4f}")
    print(f"  Mean steps   : {stats['mean_steps']:.1f}")
    print(f"  Agreement    : {stats['agreement']:.1%}")

### Σύγκριση όλων των μοντέλων (πίνακας)

In [ ]:
N_EPISODES = 100  # ← μειώστε για ταχύτητα

results = []

for algo in ALGOS:
    for grid in GRID_SIZES:
        tag = str(grid)
        grid_size = (grid, grid)
        if not model_exists(algo, grid):
            results.append({"algo": algo, "grid": grid, "status": "not trained",
                            "success": None, "reward": None, "steps": None, "agreement": None})
            continue

        print(f"Evaluating {algo.upper()} {grid}×{grid}...", end=" ", flush=True)
        if algo == "ppo":
            from stable_baselines3 import PPO
            m = PPO.load(str(MODEL_DIR / "ppo" / f"nav_ppo_{tag}"))
            predict = lambda obs, _m=m: _m.predict(obs, deterministic=True)[0]
        elif algo == "dqn":
            from stable_baselines3 import DQN
            m = DQN.load(str(MODEL_DIR / "dqn" / f"nav_dqn_{tag}"))
            predict = lambda obs, _m=m: int(_m.predict(obs, deterministic=True)[0])
        else:
            with open(MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl", "rb") as f:
                qt = pickle.load(f)
            predict = lambda obs, _qt=qt: int(np.argmax(_qt.get(_discretize_nav(obs), np.zeros(_NAV_ACTIONS))))

        eval_max_steps = max(300, grid * grid // 2)
        stats = _eval_navigation(predict, n_episodes=N_EPISODES, grid_size=grid_size, max_steps=eval_max_steps)
        results.append({"algo": algo, "grid": grid, "status": "ok",
                        "reward": stats["mean_reward"],
                        "steps": stats["mean_steps"], "agreement": stats["agreement"]})
        print(f"agreement={stats['agreement']:.1%}")

# Pretty table
print(f"\n{'Algo':<12} {'Grid':>6} {'Reward':>9} {'Steps':>8} {'Agreement':>11}")
print("-" * 50)
for r in results:
    if r["status"] == "not trained":
        print(f"{r['algo']:<12} {r['grid']:>4}×{r['grid']:<4}  {'(not trained)':>30}")
    else:
        print(f"{r['algo']:<12} {r['grid']:>4}×{r['grid']:<4}  "
              f"{r['reward']:>9.4f}  {r['steps']:>7.1f}  {r['agreement']:>10.1%}")

### Σύγκριση — bar chart

In [ ]:
trained = [r for r in results if r["status"] == "ok"]

if not trained:
    print("Δεν υπάρχουν trained models για σύγκριση.")
else:
    algo_colors = {"ppo": "tab:blue", "dqn": "tab:green", "qlearning": "tab:orange"}
    x = np.arange(len(GRID_SIZES))
    width = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, algo in enumerate(ALGOS):
        vals_agreement = []
        vals_reward = []
        for grid in GRID_SIZES:
            r = next((r for r in trained if r["algo"] == algo and r["grid"] == grid), None)
            vals_agreement.append(r["agreement"] if r else 0)
            vals_reward.append(r["reward"] if r else 0)

        offset = (i - 1) * width
        axes[0].bar(x + offset, vals_agreement, width, label=algo.upper(),
                    color=algo_colors[algo], alpha=0.85)
        axes[1].bar(x + offset, vals_reward, width, label=algo.upper(),
                    color=algo_colors[algo], alpha=0.85)

    axes[0].set_title("Agreement with Cost-Optimal Baseline", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Agreement Rate")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"{g}×{g}" for g in GRID_SIZES])
    axes[0].set_ylim(0, 1.1)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].set_title("Mean Reward", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("Mean Episode Reward")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{g}×{g}" for g in GRID_SIZES])
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.suptitle("ThessLink RL — Algorithm Comparison", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()